# Getting Started with OMJulia and the Dynawo Library

Simulate one of the Dynawo library examples from Julia, plot it, change a parameter and
simulate again.

`Dynawo.Examples.InertialGrid.DoubleInertialGrid`: two inertial grids of 32350 MVA joined by
two 120 km lines, feeding a 500 MW load. A load step at `t = 10 s` excites the inter-area
oscillation between them.

## Configuration

In [ ]:
using OMJulia
using Plots

# --- Configuration ---

# 1. Modelica Standard Library
OMLIB_DIR = joinpath(homedir(), ".openmodelica", "libraries")
ENV["OPENMODELICALIBRARY"] = OMLIB_DIR
MODELICA_PKG_PATH = joinpath(OMLIB_DIR, "Modelica 3.2.3+maint.om", "package.mo")

# 2. Dynawo Modelica Library
DYNAWO_PKG_PATH = abspath("dynawo_library/Dynawo/package.mo")

# 3. Model to simulate
MODEL = "Dynawo.Examples.InertialGrid.DoubleInertialGrid"

## Open an OpenModelica Session

In [ ]:
omc = OMJulia.OMCSession()

sendExpression(omc, "loadModel(Complex)")
sendExpression(omc, "loadModel(ModelicaServices)")

ModelicaSystem(omc, DYNAWO_PKG_PATH, MODEL, [MODELICA_PKG_PATH])

getSimulationOptions(omc)

## Simulate and Plot

`deltaFrequency` is the difference between the two frequencies, which the example computes
itself.

In [ ]:
simulate(omc)

(t, f1, f2, deltaF) = getSolutions(omc,
    ["time",
     "inertialGrid1.reducedOrderSFR.frequency",
     "inertialGrid2.reducedOrderSFR.frequency",
     "deltaFrequency"]);

In [ ]:
p1 = plot(t, [f1 f2], label = ["Inertial grid 1" "Inertial grid 2"])
plot!(p1, legend = :bottomright, titlefontsize = 12, labelfontsize = 10)
title!(p1, "Frequency of the two inertial grids")
xlabel!(p1, "Time (s)")
ylabel!(p1, "Frequency (Hz)")

In [ ]:
p2 = plot(t, deltaF, label = "deltaFrequency")
plot!(p2, legend = :bottomright, titlefontsize = 12, labelfontsize = 10)
title!(p2, "Frequency difference between the two inertial grids")
xlabel!(p2, "Time (s)")
ylabel!(p2, "deltaFrequency (Hz)")

## Change a Parameter

We change the line length and repeat the simulation.

In [ ]:
setParameters(omc, ["L1 = 150", "L2 = 150"])

simulate(omc)

(t_150, f1_150, f2_150, deltaF_150) = getSolutions(omc,
    ["time",
     "inertialGrid1.reducedOrderSFR.frequency",
     "inertialGrid2.reducedOrderSFR.frequency",
     "deltaFrequency"]);

In [ ]:
p3 = plot(t_150, [f1_150 f2_150], label = ["Inertial grid 1" "Inertial grid 2"])
plot!(p3, legend = :bottomright, titlefontsize = 12, labelfontsize = 10)
title!(p3, "Frequency of the two inertial grids, lines at 150 km")
xlabel!(p3, "Time (s)")
ylabel!(p3, "Frequency (Hz)")

In [ ]:
p4 = plot(t_150, deltaF_150, label = "deltaFrequency")
plot!(p4, legend = :bottomright, titlefontsize = 12, labelfontsize = 10)
title!(p4, "Frequency difference, lines at 150km")
xlabel!(p4, "Time (s)")
ylabel!(p4, "deltaFrequency (Hz)")

We can see oscillations before the step at t=10s. These are an effect of the model not being correctly initialized, since we mantained the operating point values of the L= 120km simulation.